In [1]:
%pip install pandas scikit-learn -q

import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = {
    "Assignment_A": "Machine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions.",
    "Assignment_B": "Machine learning is a branch of artificial intelligence that allows computers to learn from data and make predictions.",
    "Assignment_C": "Natural language processing enables computers to understand human language and analyze text using computational methods.",
    "Assignment_D": "Artificial intelligence allows machines to perform tasks that normally require human intelligence including learning reasoning and decision making.",
    "Assignment_E": "Machine learning algorithms identify patterns in data and use those patterns to make predictions and decisions.",
    "Assignment_F": "Deep learning is a subset of machine learning that uses neural networks with multiple layers to process complex data."
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

names = list(documents.keys())
texts = [clean_text(text) for text in documents.values()]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(texts)

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix * 100,
    index=names,
    columns=names
).round(2)

print("ASSIGNMENT SIMILARITY / PLAGIARISM DETECTOR")
print("=" * 60)

print("\nSIMILARITY MATRIX")
display(similarity_df)

results = []

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        score = similarity_matrix[i][j] * 100
        
        results.append({
            "Document 1": names[i],
            "Document 2": names[j],
            "Similarity (%)": round(score, 2)
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Similarity (%)",
    ascending=False
).reset_index(drop=True)

threshold = 70

results_df["Status"] = results_df["Similarity (%)"].apply(
    lambda x: "Potentially Copied" if x >= threshold else "Low Similarity"
)

print("\nRANKED DOCUMENT PAIRS")
display(results_df)

print("\nMOST SUSPICIOUS DOCUMENT PAIRS")
display(results_df.head(3))

query = input("\nEnter a new assignment text to compare: ")

query = clean_text(query)

all_texts = texts + [query]

vectorizer2 = TfidfVectorizer(stop_words="english")

matrix2 = vectorizer2.fit_transform(all_texts)

query_scores = cosine_similarity(
    matrix2[-1],
    matrix2[:-1]
)[0]

query_results = pd.DataFrame({
    "Assignment": names,
    "Similarity (%)": [
        round(score * 100, 2)
        for score in query_scores
    ]
})

query_results["Status"] = query_results["Similarity (%)"].apply(
    lambda x: "Potentially Copied" if x >= threshold else "Low Similarity"
)

query_results = query_results.sort_values(
    "Similarity (%)",
    ascending=False
).reset_index(drop=True)

print("\nNEW ASSIGNMENT COMPARISON")
display(query_results)

print("\nMOST SIMILAR ASSIGNMENT:")
print(query_results.iloc[0]["Assignment"])

print(
    "Similarity:",
    query_results.iloc[0]["Similarity (%)"],
    "%"
)

print(
    "Status:",
    query_results.iloc[0]["Status"]
)

Note: you may need to restart the kernel to use updated packages.
ASSIGNMENT SIMILARITY / PLAGIARISM DETECTOR

SIMILARITY MATRIX


,Assignment_A,Assignment_B,Assignment_C,Assignment_D,Assignment_E,Assignment_F
Assignment_A,100.00,87.50,13.35,20.33,26.34,16.16
Assignment_B,87.50,100.00,5.56,28.37,26.34,16.16
Assignment_C,13.35,5.56,100.00,5.02,0.00,0.00
Assignment_D,20.33,28.37,5.02,100.00,2.31,4.44
Assignment_E,26.34,26.34,0.00,2.31,100.00,11.89
Assignment_F,16.16,16.16,0.00,4.44,11.89,100.00



RANKED DOCUMENT PAIRS


,Document 1,Document 2,Similarity (%),Status
0,Assignment_A,Assignment_B,87.50,Potentially Copied
1,Assignment_B,Assignment_D,28.37,Low Similarity
2,Assignment_A,Assignment_E,26.34,Low Similarity
3,Assignment_B,Assignment_E,26.34,Low Similarity
4,Assignment_A,Assignment_D,20.33,Low Similarity
5,Assignment_B,Assignment_F,16.16,Low Similarity
6,Assignment_A,Assignment_F,16.16,Low Similarity
7,Assignment_A,Assignment_C,13.35,Low Similarity
8,Assignment_E,Assignment_F,11.89,Low Similarity
9,Assignment_B,Assignment_C,5.56,Low Similarity



MOST SUSPICIOUS DOCUMENT PAIRS


,Document 1,Document 2,Similarity (%),Status
0,Assignment_A,Assignment_B,87.50,Potentially Copied
1,Assignment_B,Assignment_D,28.37,Low Similarity
2,Assignment_A,Assignment_E,26.34,Low Similarity



Enter a new assignment text to compare:  Machine learning is a branch of artificial intelligence that enables computers to learn from data and make predictions.



NEW ASSIGNMENT COMPARISON


,Assignment,Similarity (%),Status
0,Assignment_A,100.00,Potentially Copied
1,Assignment_B,86.18,Potentially Copied
2,Assignment_E,24.67,Low Similarity
3,Assignment_D,18.79,Low Similarity
4,Assignment_F,15.55,Low Similarity
5,Assignment_C,11.67,Low Similarity



MOST SIMILAR ASSIGNMENT:
Assignment_A
Similarity: 100.0 %
Status: Potentially Copied
